# Health Recommendation System — Multi-Label Classification: Model Comparison

This notebook trains and evaluates three classifiers on the task of predicting health recommendations from numeric health dimension inputs.

**Problem formulation:**
- **Input (X):** 11 numeric features — midpoint-encoded values of 8 health dimensions (`dif_*` and `c_val_*`)
- **Output (Y):** 53 binary labels — 29 tip presence flags (Y1) + 24 status flags across 8 dimensions × 3 states (Y2)
- **Task:** Multi-label classification

**Models compared:**
| Model | Hamming Loss | F1 (Micro) | Training Time |
|-------|-------------|------------|---------------|
| XGBoost | 0.1170 | **0.7681** | — |
| Random Forest | 0.1237 | 0.7611 | 1698.5s |
| Logistic Regression | 0.1466 | 0.6984 | 188.0s |

---
> ⚠️ **Data Notice:** The original dataset is proprietary and cannot be shared publicly.  
> A mock sample (`data_sample.xlsx`) with identical column structure is provided for reproduction.  
> **Outputs below were produced on the full dataset (746,496 rows) and are preserved for reference.**


---
## 1 — Data Preparation

The raw dataset stores all health dimension values as range strings (e.g. `"(-250)-0"`).  
These are decoded to numeric midpoints to form the feature matrix X.

Two target matrices are constructed using sparse encoding:
- **Y1** (29 columns): tip presence, detected via keyword matching on the `recommendations` column
- **Y2** (24 columns): dimension status flags, parsed from the `status_assessment` column (8 dimensions × 3 states: Improving / Acceptable / Not Mentioned)

Both are combined into a single sparse label matrix Y of shape `(746496, 53)`.


In [ ]:
import pickle

with open('final_base_tips.pkl', 'rb') as f:
    updated_sentences = pickle.load(f)

tip_list = sorted(list(updated_sentences))

print(f"Base Tips restored successfully. Total count: {len(tip_list)}")
for i, t in enumerate(tip_list[:5]):
    print(f"{i+1}: {t}")

Base Tips restored successfully. Total count: 29
1: Anticipate and plan for the challenges that may arise
2: Avoid harmful behaviors, such as smoking and excessive drinking
3: Avoid who stress you out: If someone is constantly causing stress in your  life and you cannot break the relationship, limit the time you spend with it, or break the relationship completely if possible
4: Change your mindset Try to look at problems from a positive perspective For example, when you’re stuck in traffic, look at it as an opportunity to pause and listen to your favorite radio station
5: Chew gum or eat mints when you feel the urge to smoke


In [ ]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import coo_matrix, hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import gc

# ==========================================
# 1. Load data & basic filtering
# ==========================================
file_path = 'data_sample.xlsx'
cols_to_drop = ['dif_nutrition', 'c_val_nutrition', 'dif_obesity', 'c_val_obesity', 'dif_sleep', 'Unnamed: 0','Unnamed: 17']

print("Loading data...")
df = pd.read_excel(file_path, usecols=lambda x: x not in cols_to_drop and not x.startswith('Unnamed'))

# ==========================================
# 2. Midpoint Encoding for Range columns (Feature X)
# ==========================================

def range_to_midpoint(range_str):
    s = str(range_str).strip()
    clean = s.replace('(', '').replace(')', '')
    parts = re.split(r'(?<=\d)-', clean, maxsplit=1)
    if len(parts) == 2:
        low = float(parts[0])
        high = float(parts[1])
        return (low + high) / 2
    return np.nan

x_cols = df.columns.difference(['recommendations', 'status_assessment'])

X_numeric = pd.DataFrame(index=df.index)
for col in x_cols:
    X_numeric[col] = df[col].apply(range_to_midpoint)

X_numeric = X_numeric.astype(np.float32)

print("Midpoint encoding for Range columns completed.")

# ==========================================
# 3. Process Y1: recommendations (29 Tips)
# ==========================================
print("Processing target variable Y1 (29 Tips)...")
tips_keywords = [
    "Anticipate and plan", "Avoid harmful behaviors", "Avoid who stress you out",
    "Change your mindset", "Chew gum", "Do things that make you happy",
    "Don't spend too much time", "Drink plenty of water", "Get a journal",
    "Get at least seven hours", "Increase your intake of NSP", "Increasing levels of physical activity",
    "Keep your hands busy", "Learn to say no", "Making a schedule",
    "Pick up a nice plushy", "Play a sport", "Read about the harmful effects",
    "Realize that your situation", "Refrain from using electronics", "Restrict the intake of sugars",
    "Set attainable goals", "Share your feelings", "Take a post-meal walk",
    "Tell your family and friends", "Try to exercise at least 3 times", "Try to pick up some hobbies",
    "When facing great challenges", "engage in at least 30 minutes"
]

row_idx_y1, col_idx_y1 = [], []
recs_series = df['recommendations'].astype(str)
for j, keyword in enumerate(tips_keywords):
    mask = recs_series.str.contains(keyword, regex=False, na=False).values
    rows = np.where(mask)[0]
    row_idx_y1.extend(rows)
    col_idx_y1.extend(np.full_like(rows, j))

Y1_sparse = coo_matrix((np.ones(len(row_idx_y1), dtype=np.int8),
                       (row_idx_y1, col_idx_y1)),
                       shape=(len(df), len(tips_keywords)))

# ==========================================
# 4. Process Y2: status_assessment (8 dimensions × 3 states)
# ==========================================
print("Processing target variable Y2 (24 status flags)...")
dimensions = ['Obesity', 'Sleep', 'Nutrition', 'Movement', 'Anti-Smoke', 'Anti-Stress', 'Wellness', 'Depression']
parts = df['status_assessment'].astype(str).str.split('acceptable level for', n=1, expand=True)
part_imp = parts[0].fillna('')
part_acc = parts[1].fillna('')

row_idx_y2, col_idx_y2 = [], []
for i, dim in enumerate(dimensions):
    is_imp = part_imp.str.contains(dim, regex=False).values
    is_acc = part_acc.str.contains(dim, regex=False).values
    is_not = ~(is_imp | is_acc)

    for offset, mask in enumerate([is_imp, is_acc, is_not]):
        rows = np.where(mask)[0]
        row_idx_y2.extend(rows)
        col_idx_y2.extend(np.full_like(rows, i * 3 + offset))

Y2_sparse = coo_matrix((np.ones(len(row_idx_y2), dtype=np.int8),
                       (row_idx_y2, col_idx_y2)),
                       shape=(len(df), len(dimensions) * 3))

# ==========================================
# 5. Combine features, free memory, split dataset
# ==========================================
Y_final = hstack([Y1_sparse, Y2_sparse]).tocsr()

print("\n--- Preview of Feature Matrix X (Midpoint Encoded) - First 5 Rows ---")
print(X_numeric.head())

X_final = X_numeric.values
del df, parts, part_imp, part_acc, X_numeric
gc.collect()
print("\nOriginal text data cleared from memory.")

print("Splitting into training and test sets...")
X_train, X_test, y_train, y_test = train_test_split(X_final, Y_final, test_size=0.2, random_state=42)

print("========================================")
print(f"Training set X shape: {X_train.shape}")
print(f"Training set Y shape: {y_train.shape} (multi-label)")
print(f"Test set X shape: {X_test.shape}")
print("========================================")
print("Data preparation completed! Ready for model training.")

Loading data...
Midpoint encoding for Range columns completed.
Processing target variable Y1 (29 Tips)...
Processing target variable Y2 (24 status flags)...

--- Preview of Feature Matrix X (Midpoint Encoded) - First 5 Rows ---
   c_val_anti_smoke  c_val_anti_stress  c_val_depression  c_val_movement  \
0             200.0              200.0             200.0           200.0   
1             200.0              200.0             200.0           500.0   
2             200.0              200.0             200.0           800.0   
3             200.0              200.0             200.0           200.0   
4             200.0              200.0             200.0           500.0   

   c_val_sleep  c_val_wellness  dif_anti_smoke  dif_anti_stress  \
0        200.0           200.0           125.0            125.0   
1        200.0           200.0           125.0            125.0   
2        200.0           200.0           125.0            125.0   
3        200.0           200.0           125.0 

---
## 2 — Model 1: XGBoost

XGBoost with `MultiOutputClassifier` wrapping — trains one XGBoost tree per label column in parallel.  
8 label columns with zero variance are skipped (constant across all training rows).

**Hyperparameters:** `n_estimators=100`, `max_depth=6`, `learning_rate=0.1`, `tree_method='hist'`


In [ ]:
import sys
!{sys.executable} -m pip install -q xgboost

In [ ]:
import xgboost
from xgboost import XGBClassifier
print("xgboost version:", xgboost.__version__)

xgboost version: 3.2.0


In [ ]:
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import hamming_loss, f1_score
import numpy as np

# 1. Convert to dense arrays for easier processing
y_train_dense = y_train.toarray()
y_test_dense = y_test.toarray()

# 2. Identify columns that contain both 0 and 1
# Columns that are all 0s or all 1s cannot be trained by XGBoost
valid_cols = []
for i in range(y_train_dense.shape[1]):
    unique_values = np.unique(y_train_dense[:, i])
    if len(unique_values) > 1:
        valid_cols.append(i)
    else:
        print(f"Skipping column {i}: this column cannot be trained because it contains only {unique_values}.")

# Keep only valid target columns for training
y_train_filtered = y_train_dense[:, valid_cols]
y_test_filtered = y_test_dense[:, valid_cols]

print(f"After filtering, number of valid label columns: {len(valid_cols)} / 53")

# 3. Configure base XGBoost model
base_xgb = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist',
    random_state=42
)

# 4. Wrap with MultiOutputClassifier for multi-label classification
# If parallel training fails, reduce n_jobs (e.g., n_jobs=4)
multilabel_model = MultiOutputClassifier(base_xgb, n_jobs=-1)

# 5. Train model
print("Training sub-models on valid label columns...")
multilabel_model.fit(X_train, y_train_filtered)
print("Training completed!")

# 6. Prediction & evaluation
y_pred = multilabel_model.predict(X_test)
h_loss = hamming_loss(y_test_filtered, y_pred)
f1_micro = f1_score(y_test_filtered, y_pred, average='micro')

print("\n========================================")
print(f"Valid label count: {len(valid_cols)}")
print(f"Test Hamming Loss: {h_loss:.4f}")
print(f"Test F1 Score (Micro): {f1_micro:.4f}")
print("========================================")

Skipping column 6: this column cannot be trained because it contains only [0].
Skipping column 29: this column cannot be trained because it contains only [1].
Skipping column 30: this column cannot be trained because it contains only [0].
Skipping column 31: this column cannot be trained because it contains only [0].
Skipping column 34: this column cannot be trained because it contains only [0].
Skipping column 35: this column cannot be trained because it contains only [0].
Skipping column 36: this column cannot be trained because it contains only [1].
Skipping column 37: this column cannot be trained because it contains only [0].
After filtering, number of valid label columns: 45 / 53
Training sub-models on valid label columns...
Training completed!

Valid label count: 45
Test Hamming Loss: 0.1170
Test F1 Score (Micro): 0.7681


---
## 3 — Model 2: Random Forest

Random Forest trained label-by-label (one classifier per output column) to manage memory on the full dataset.  
Training time reflects the cost of fitting 45 separate tree ensembles on 597,196 rows.

**Hyperparameters:** `n_estimators=100`, `max_depth=None`, `n_jobs=2`


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import hamming_loss, f1_score
import numpy as np
import time

# 1. Filter out columns with a single category (using sparse format, without converting to dense).
y_train_csc = y_train.tocsc()
y_test_csc = y_test.tocsc()

valid_cols = []
for i in range(y_train_csc.shape[1]):
    col_data = y_train_csc[:, i].toarray().ravel()
    if len(np.unique(col_data)) > 1:
        valid_cols.append(i)
    else:
        print(f"Skip column {i}: Single category, cannot be trained.")

print(f"After filtering, the actual number of labels used in training: {len(valid_cols)} / 53")

# 2. Training Random Forest label by label (avoiding loading all labels into memory at once)
print("Training the Random Forest model label by label...")
start_time = time.time()

y_pred_all = np.zeros((X_test.shape[0], len(valid_cols)), dtype=np.int8)

for idx, col_i in enumerate(valid_cols):
    y_col_train = y_train_csc[:, col_i].toarray().ravel()

    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=2
    )
    rf.fit(X_train, y_col_train)
    y_pred_all[:, idx] = rf.predict(X_test)

    if (idx + 1) % 10 == 0:
        print(f"  already done {idx + 1}/{len(valid_cols)} tags...")

rf_train_time = time.time() - start_time
print(f"Training complete! Time taken: {rf_train_time:.1f}s")

# 3. Assessment
y_test_filtered = y_test_csc[:, valid_cols].toarray()
rf_h_loss = hamming_loss(y_test_filtered, y_pred_all)
rf_f1_micro = f1_score(y_test_filtered, y_pred_all, average='micro')

print(f"\n========================================")
print(f"Random Forest Results (Midpoint Encoding)")
print(f"Valid tag count: {len(valid_cols)}")
print(f"Trainset Hamming Loss: {rf_h_loss:.4f}")
print(f"Testset F1 Score (Micro): {rf_f1_micro:.4f}")
print(f"Training time: {rf_train_time:.1f}s")
print(f"========================================")

Skip column 6: Single category, cannot be trained.
Skip column 29: Single category, cannot be trained.
Skip column 30: Single category, cannot be trained.
Skip column 31: Single category, cannot be trained.
Skip column 34: Single category, cannot be trained.
Skip column 35: Single category, cannot be trained.
Skip column 36: Single category, cannot be trained.
Skip column 37: Single category, cannot be trained.
After filtering, the actual number of labels used in training: 45 / 53
Training the Random Forest model label by label...
  already done 10/45 tags...
  already done 20/45 tags...
  already done 30/45 tags...
Training complete! Time taken: 1698.5s

Random Forest Results (Midpoint Encoding)
Valid tag count: 45
Trainset Hamming Loss: 0.1237
Testset F1 Score (Micro): 0.7611
Training time: 1698.5s


---
## 4 — Model 3: Logistic Regression

Logistic Regression as a linear baseline, also trained label-by-label.  
Substantially faster than tree-based models, at the cost of lower F1.

**Hyperparameters:** `max_iter=1000`, `solver='lbfgs'`


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import hamming_loss, f1_score
import numpy as np
import time

# 1. Filter out columns with a single category
y_train_csc = y_train.tocsc()
y_test_csc = y_test.tocsc()

valid_cols = []
for i in range(y_train_csc.shape[1]):
    col_data = y_train_csc[:, i].toarray().ravel()
    if len(np.unique(col_data)) > 1:
        valid_cols.append(i)
    else:
        print(f"Skip column {i}: Single category, cannot be trained.")

print(f"After filtering, the actual number of labels used in training: {len(valid_cols)} / 53")

# 2. Label-by-label training of Logistic Regression
print("Training a Logistic Regression model label by label...")
start_time = time.time()

y_pred_all = np.zeros((X_test.shape[0], len(valid_cols)), dtype=np.int8)

for idx, col_i in enumerate(valid_cols):
    y_col_train = y_train_csc[:, col_i].toarray().ravel()

    lr = LogisticRegression(
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    )
    lr.fit(X_train, y_col_train)
    y_pred_all[:, idx] = lr.predict(X_test)

    if (idx + 1) % 10 == 0:
        print(f"  already done {idx + 1}/{len(valid_cols)} tags...")

lr_train_time = time.time() - start_time
print(f"Training complete! Time taken: {lr_train_time:.1f}s")

# 3. Assessment
y_test_filtered = y_test_csc[:, valid_cols].toarray()
lr_h_loss = hamming_loss(y_test_filtered, y_pred_all)
lr_f1_micro = f1_score(y_test_filtered, y_pred_all, average='micro')

print(f"\n========================================")
print(f"Logistic Regression Results (Midpoint Encoding)")
print(f"Valid tag count: {len(valid_cols)}")
print(f"trainset Hamming Loss: {lr_h_loss:.4f}")
print(f"testset F1 Score (Micro): {lr_f1_micro:.4f}")
print(f"Training time:: {lr_train_time:.1f}s")
print(f"========================================")

Skip column 6: Single category, cannot be trained.
Skip column 29: Single category, cannot be trained.
Skip column 30: Single category, cannot be trained.
Skip column 31: Single category, cannot be trained.
Skip column 34: Single category, cannot be trained.
Skip column 35: Single category, cannot be trained.
Skip column 36: Single category, cannot be trained.
Skip column 37: Single category, cannot be trained.
After filtering, the actual number of labels used in training: 45 / 53
Training a Logistic Regression model label by label...
  already done 10/45 tags...
  already done 20/45 tags...
  already done 30/45 tags...
  already done 40/45 tags...
Training complete! Time taken: 188.0s

Logistic Regression Results (Midpoint Encoding)
Valid tag count: 45
trainset Hamming Loss: 0.1466
testset F1 Score (Micro): 0.6984
Training time:: 188.0s


---
## 5 — Model Comparison & Analysis

| Model | Hamming Loss | F1 (Micro) | Training Time |
|-------|-------------|------------|---------------|
| **XGBoost** | **0.1170** | **0.7681** | — |
| Random Forest | 0.1237 | 0.7611 | 1698.5s |
| Logistic Regression | 0.1466 | 0.6984 | 188.0s |

**Key observations:**

- **XGBoost** achieves the best F1 (0.7681) and lowest Hamming Loss (0.1170), making it the strongest overall performer on this multi-label task
- **Random Forest** is competitive (F1 0.7611) but ~28× slower than Logistic Regression due to label-by-label tree fitting across 45 output columns
- **Logistic Regression** is the fastest and most interpretable baseline, but F1 drops ~7 points vs XGBoost — suggesting the decision boundaries between health dimension ranges are non-linear
- All three models handle the multi-label structure well; the main differentiator is predictive power on minority labels (low-frequency tips)
